In [1]:
from pathlib import Path

def get_text_file(path):
    with open(path, "r") as f:
        return f.read()

In [4]:
import glob
text_files = sorted(glob.glob("./data/text/*.txt"))
text_files

['./data/text/04_Alfonso_Tamay.txt',
 './data/text/05_Felipe_May.txt',
 './data/text/06_Gricelda_Pech.txt',
 './data/text/07_Eligio_Uicab_Tomojchi.txt',
 './data/text/08_Teodoro_May.txt',
 './data/text/09_Adolfo_Chuc.txt',
 './data/text/10_Jesus_Euan.txt',
 './data/text/11_Hector_May.txt',
 './data/text/12_Lourdes_y_Marcela_Ucam.txt',
 './data/text/13_Venustiano_Puc.txt',
 './data/text/14_Micaela_Ek.txt',
 './data/text/15_Mario_Chan.txt']

In [5]:
for file in text_files:
    text = get_text_file(file).split("\n\n")
    file_path = Path(file)
    audios = glob.glob(f"./3h-audios/{file_path.stem}*.wav")
    print(f"{file} - {len(text)} utterances - {len(audios)} audios")

./data/text/04_Alfonso_Tamay.txt - 66 utterances - 66 audios
./data/text/05_Felipe_May.txt - 81 utterances - 81 audios
./data/text/06_Gricelda_Pech.txt - 43 utterances - 43 audios
./data/text/07_Eligio_Uicab_Tomojchi.txt - 50 utterances - 50 audios
./data/text/08_Teodoro_May.txt - 123 utterances - 123 audios
./data/text/09_Adolfo_Chuc.txt - 64 utterances - 64 audios
./data/text/10_Jesus_Euan.txt - 60 utterances - 60 audios
./data/text/11_Hector_May.txt - 31 utterances - 31 audios
./data/text/12_Lourdes_y_Marcela_Ucam.txt - 56 utterances - 56 audios
./data/text/13_Venustiano_Puc.txt - 140 utterances - 140 audios
./data/text/14_Micaela_Ek.txt - 162 utterances - 162 audios
./data/text/15_Mario_Chan.txt - 227 utterances - 227 audios


In [6]:
import pandas as pd

spk_metadata = pd.read_csv("./data/spk_metadata.csv")
spk_metadata[spk_metadata[" description"].str.contains("Eligio")].iloc[0]

def get_spk_id(row):
    if row["spk_id"].startswith("spk"):
        return row["spk_id"]
    
    name = row["spk_id"][3:10]
    info_spk = spk_metadata[spk_metadata[" description"].str.contains(name)].iloc[0]
    new_spk_id = info_spk["spk_id"]
    
    return new_spk_id

In [8]:
df_first_part = pd.read_csv("./data/3h-corrected-transcripts-manual.csv")
df_first_part.head(5)

,file_name,mms,correction,found
0,01_Anatolio_Pech,le tzicbalob nu kaaba'e xta'cun bixunan xta'cu...,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,yes
1,01_Anatolio_Pech,a ta' kun vi xonaano le xonaano' jun p'ee co'l...,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,yes
2,01_Anatolio_Pech,u kaaba'e' ya'ax che' paalo meeque ya'ala' ti'...,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,yes
3,01_Anatolio_Pech,caan yila' t nupital u yaabitale' quyoco ti ch...,káan u yila tu bin tu taal u yáa'biltale ku yo...,yes
4,01_Anatolio_Pech,pero leti' utucul beyo le can wenequele u yumi...,pero leti'e u tuukul beyo le káan weenek le u ...,yes


In [10]:
df_first_part_filtered = df_first_part[df_first_part["file_name"].isin([
    "01_Anatolio_Pech",
    "02_Liboria_May",
    "03_Eligio_Uicab_Jatswooj"
])]

df_first_part_filtered.head(5)

,file_name,mms,correction,found
0,01_Anatolio_Pech,le tzicbalob nu kaaba'e xta'cun bixunan xta'cu...,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,yes
1,01_Anatolio_Pech,a ta' kun vi xonaano le xonaano' jun p'ee co'l...,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,yes
2,01_Anatolio_Pech,u kaaba'e' ya'ax che' paalo meeque ya'ala' ti'...,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,yes
3,01_Anatolio_Pech,caan yila' t nupital u yaabitale' quyoco ti ch...,káan u yila tu bin tu taal u yáa'biltale ku yo...,yes
4,01_Anatolio_Pech,pero leti' utucul beyo le can wenequele u yumi...,pero leti'e u tuukul beyo le káan weenek le u ...,yes


In [11]:
import librosa

file_name_list = ["01_Anatolio_Pech", "02_Liboria_May", "03_Eligio_Uicab_Jatswooj"]

utterances = []

for filename in file_name_list:  
    print(filename)      
    rows = df_first_part[df_first_part["file_name"] == filename].iterrows()
    for idx, (_, row) in enumerate(rows):
        audio_path = f"./3h-audios/{row["file_name"]}_{idx+1}.wav"
        audio, sr = librosa.load(audio_path, sr=16000)
        duration = librosa.get_duration(y=audio, sr=sr)
        utt = {
            "utt_id": f"{row["file_name"]}_{idx+1}",
            "maya": row["correction"],
            "spk_id": filename
        }

        utterances.append(utt)

01_Anatolio_Pech
02_Liboria_May
03_Eligio_Uicab_Jatswooj


In [12]:
for file in text_files:
    text_list = get_text_file(file).split("\n\n")
    file_name = Path(file).stem

    for idx, texto in enumerate(text_list):
        audio_path = f"./3h-audios/{file_name}_{idx+1}.wav"
        audio, sr = librosa.load(audio_path, sr=16000)
        duration = librosa.get_duration(y=audio, sr=sr)
        utt = {
            "utt_id": f"{file_name}_{idx+1}",
            "maya": texto.strip(),
            "spk_id": file_name
        }

        utterances.append(utt)

In [13]:
df_3h = pd.DataFrame(utterances)

In [14]:
df_3h.head(5)


,utt_id,maya,spk_id
0,01_Anatolio_Pech_1,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,01_Anatolio_Pech
1,01_Anatolio_Pech_2,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,01_Anatolio_Pech
2,01_Anatolio_Pech_3,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,01_Anatolio_Pech
3,01_Anatolio_Pech_4,káan u yila tu bin tu taal u yáa'biltale ku yo...,01_Anatolio_Pech
4,01_Anatolio_Pech_5,pero leti'e u tuukul beyo le káan weenek le u ...,01_Anatolio_Pech


In [15]:
df_3h["new_spk_id"] = df_3h.apply(lambda x: get_spk_id(x), axis=1)

df_3h[df_3h["new_spk_id"] == "spk_021"]

,utt_id,maya,spk_id,new_spk_id
77,03_Eligio_Uicab_Jatswooj_1,le wa a k'áat ka in tsikbalt teech maaya leti'...,03_Eligio_Uicab_Jatswooj,spk_021
78,03_Eligio_Uicab_Jatswooj_2,ja'asajáóol je'elo ku jook'olo'ob k'áaxo yaan ...,03_Eligio_Uicab_Jatswooj,spk_021
79,03_Eligio_Uicab_Jatswooj_3,ku báalantikuba'ob tu paach le che xano utia'a...,03_Eligio_Uicab_Jatswooj,spk_021
80,03_Eligio_Uicab_Jatswooj_4,je'elo le túun le óotsil máako'obo leti'obe ko...,03_Eligio_Uicab_Jatswooj,spk_021
81,03_Eligio_Uicab_Jatswooj_5,yúuntun le yúuntune le ku pi'ik'tiko'obe ku ts...,03_Eligio_Uicab_Jatswooj,spk_021
...,...,...,...,...
359,07_Eligio_Uicab_Tomojchi_46,entonkes leti'obe tu tuklo'ob leti'obe ma tu k...,07_Eligio_Uicab_Tomojchi,spk_021
360,07_Eligio_Uicab_Tomojchi_47,jkíimo'obe es ke ka tu yu'ubo'ob u taal leti l...,07_Eligio_Uicab_Tomojchi,spk_021
361,07_Eligio_Uicab_Tomojchi_48,leti'obe jela'an u yiliko'obe tumen leti'obe y...,07_Eligio_Uicab_Tomojchi,spk_021
362,07_Eligio_Uicab_Tomojchi_49,wi'it'o'ob leti'ob chéen le jaaj u k'axmaj u k...,07_Eligio_Uicab_Tomojchi,spk_021


In [16]:
df_3h["utt_num"] = df_3h.groupby("new_spk_id").cumcount() + 1

df_3h.head(5)

,utt_id,maya,spk_id,new_spk_id,utt_num
0,01_Anatolio_Pech_1,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,01_Anatolio_Pech,spk_019,1
1,01_Anatolio_Pech_2,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,01_Anatolio_Pech,spk_019,2
2,01_Anatolio_Pech_3,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,01_Anatolio_Pech,spk_019,3
3,01_Anatolio_Pech_4,káan u yila tu bin tu taal u yáa'biltale ku yo...,01_Anatolio_Pech,spk_019,4
4,01_Anatolio_Pech_5,pero leti'e u tuukul beyo le káan weenek le u ...,01_Anatolio_Pech,spk_019,5


In [17]:
df_3h["new_id"] = df_3h["new_spk_id"] + "_utt_" + df_3h["utt_num"].astype(str).str.zfill(4)

df_3h_clean = df_3h.drop(columns=["spk_id"]).rename(columns={
    "new_id":"utt_id",
    "maya":"maya",
    "new_spk_id":"spk_id",
    "utt_id":"filename",
})

In [18]:
df_3h_clean = df_3h_clean[["utt_id", "maya", "spk_id", "filename"]]
df_3h_clean["utt_id"].value_counts()

utt_id
spk_019_utt_0001    1
spk_019_utt_0002    1
spk_019_utt_0003    1
spk_019_utt_0004    1
spk_019_utt_0005    1
                   ..
spk_032_utt_0223    1
spk_032_utt_0224    1
spk_032_utt_0225    1
spk_032_utt_0226    1
spk_032_utt_0227    1
Name: count, Length: 1227, dtype: int64

In [19]:
from kinai.classes.paths import Paths


paths = Paths()

df_1h = pd.read_csv(str(paths.annotations / "data.csv"))
df_1h.head(5)

,utt_id,maya,spanish,spk_id,start,end
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050


In [20]:
# crear nuevo id

df_1h["utt_num"] = df_1h.groupby("spk_id").cumcount() + 1
df_1h["new_id"] = df_1h["spk_id"] + "_utt_" + df_1h["utt_num"].astype(str).str.zfill(4)

df_1h.head(5)

,utt_id,maya,spanish,spk_id,start,end,utt_num,new_id
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450,1,spk_001_utt_0001
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250,2,spk_001_utt_0002
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350,3,spk_001_utt_0003
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550,4,spk_001_utt_0004
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050,5,spk_001_utt_0005


In [21]:
df_1h_clean = df_1h.rename(columns={
    "new_id":"utt_id",
    "maya":"maya",
    "spk_id":"spk_id",
    "utt_id":"filename",
})

df_1h_clean = df_1h_clean[["utt_id", "maya", "spk_id", "filename"]]

df_1h_clean.head(5)

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004


In [22]:
df_all = pd.concat([df_1h_clean, df_3h_clean], ignore_index=True)
df_all.head(5)

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004


In [23]:
df_all[df_all["utt_id"] == "spk_021_utt_0030"]

,utt_id,maya,spk_id,filename
1414,spk_021_utt_0030,ka tu ya'alajo'obe bejla'e ma ki bejla'e ma...,spk_021,03_Eligio_Uicab_Jatswooj_30


In [ ]:
df_all.to_csv("./data/dataset.csv", index=False)

In [25]:
df_all

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004
...,...,...,...,...
2530,spk_032_utt_0223,jumpuul osea ma jumpuuli cada administración p...,spk_032,15_Mario_Chan_223
2531,spk_032_utt_0224,como to'on láaj explicado to'on beeta'an to'on...,spk_032,15_Mario_Chan_224
2532,spk_032_utt_0225,le ku xkáakpachtiko'obe ma tu pak'o'obi ma tu ...,spk_032,15_Mario_Chan_225
2533,spk_032_utt_0226,teen xan man ti'ob weye camionadasil arroz kin...,spk_032,15_Mario_Chan_226
